In [ ]:
import pandas as pd
import numpy as np
from ctgan import CTGAN
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# ---------------------------------------------------------
# 1. LOAD DỮ LIỆU & CẤU HÌNH (GIỮ NGUYÊN)
# ---------------------------------------------------------
print("Đang đọc dữ liệu...")
data = pd.read_csv('Agri_Data_Cleaned.csv')

# --- QUAN TRỌNG: Loại bỏ Yield để CTGAN không học cột này ---
if 'Yield' in data.columns:
    data = data.drop(columns=['Yield'])
    print("Đã loại bỏ cột 'Yield' khỏi dữ liệu huấn luyện.")

# Tự động phát hiện metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data)

discrete_columns = [
    'District', 'Season', 'Crop Name', 'Transplant', 
    'Growth', 'Harvest', 'pH_Suitability', 
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 
    'is_extreme_Wind_Max'
]

# ---------------------------------------------------------
# 2. HUẤN LUYỆN CTGAN (GIỮ NGUYÊN KIẾN TRÚC TUNING)
# ---------------------------------------------------------
print("Đang khởi tạo CTGAN với cấu hình tối ưu...")

# Giữ nguyên siêu tham số bạn yêu cầu
ctgan = CTGAN(
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    batch_size=500,
    epochs=500,
    verbose=True
)

print("Bắt đầu huấn luyện... (Quá trình này có thể mất vài phút)")
ctgan.fit(data, discrete_columns)

# ---------------------------------------------------------
# 3. SINH DỮ LIỆU (REJECT SAMPLING)
# ---------------------------------------------------------
print("Đang sinh dữ liệu theo chiến lược 'Reject Sampling'...")
target_rows = 20000
buffer_rows = int(target_rows * 1.3) 

synthetic_data_raw = ctgan.sample(buffer_rows)

# --- BỘ LỌC CHẤT LƯỢNG SƠ CẤP ---
mask_valid = (
    (synthetic_data_raw['Area'] > 0) &
    (synthetic_data_raw['Production'] >= 0) &
    (synthetic_data_raw['Rainfall'] >= 0) &
    (synthetic_data_raw['Avg Temp'] > 5) & (synthetic_data_raw['Avg Temp'] < 50)
)
synthetic_data = synthetic_data_raw[mask_valid].copy()

if len(synthetic_data) > target_rows:
    synthetic_data = synthetic_data.sample(n=target_rows, random_state=42)
else:
    missing = target_rows - len(synthetic_data)
    extra = synthetic_data.sample(n=missing, replace=True)
    synthetic_data = pd.concat([synthetic_data, extra])

print(f"Đã chọn lọc {len(synthetic_data)} mẫu chất lượng cao.")

# --- RESET INDEX & FIX DTYPE ---
synthetic_data = synthetic_data.reset_index(drop=True)

cols_to_float = [
    'Min Temp', 'Max Temp', 'Avg Temp', 
    'Min Relative Humidity', 'Max Relative Humidity', 
    'Heat_Stress_Days', 'Wind_Max', 'Wind_Mean'
]
for col in cols_to_float:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].astype(float)

# =========================================================
# 4. HẬU XỬ LÝ NÂNG CAO (DISTRIBUTION MATCHING) - MỚI!
# =========================================================
print("Đang áp dụng thuật toán Quantile Matching để sửa phân phối Area & Production...")

def impose_distribution(synthetic_series, real_series):
    """
    Ép phân phối dữ liệu giả theo dữ liệu thật (Quantile Mapping)
    nhưng giữ nguyên thứ tự (Rank) để bảo toàn Correlation của CTGAN.
    """
    # 1. Tính rank chuẩn hóa của dữ liệu giả [0, 1]
    n_synthetic = len(synthetic_series)
    # method='average' xử lý các giá trị trùng nhau
    ranks = synthetic_series.rank(method='average').values 
    ranks_norm = (ranks - 1) / (n_synthetic - 1)
    
    # 2. Lấy phân phối mẫu từ dữ liệu thật
    real_sorted = np.sort(real_series.dropna().values)
    n_real = len(real_sorted)
    
    # 3. Tạo trục phân vị cho dữ liệu thật
    real_quantiles = np.linspace(0, 1, n_real)
    
    # 4. Ánh xạ giá trị (Nội suy)
    # Tìm giá trị thật tương ứng với mức rank của giá trị giả
    mapped_values = np.interp(ranks_norm, real_quantiles, real_sorted)
    
    return mapped_values

# --- ÁP DỤNG CHO AREA VÀ PRODUCTION ---
# Bước này cực kỳ quan trọng để KS Statistic và Mean Diff giảm xuống mức tối thiểu
if 'Area' in synthetic_data.columns:
    mapped_area = impose_distribution(synthetic_data['Area'], data['Area'])
    # Làm tròn về int vì Area là số nguyên
    synthetic_data['Area'] = np.round(mapped_area).astype(int)

if 'Production' in synthetic_data.columns:
    mapped_prod = impose_distribution(synthetic_data['Production'], data['Production'])
    synthetic_data['Production'] = np.round(mapped_prod).astype(int)

# Đảm bảo không có giá trị <= 0 sau khi map (dù ít khả năng xảy ra)
synthetic_data['Area'] = synthetic_data['Area'].clip(lower=1)
synthetic_data['Production'] = synthetic_data['Production'].clip(lower=0)


# =========================================================
# 5. HẬU XỬ LÝ LOGIC (LOGIC ENFORCEMENT - GIỮ NGUYÊN)
# =========================================================
print("Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...")

# --- A. ĐỒNG BỘ CÂY TRỒNG & THỜI GIAN ---
cols_to_sync = ['Season', 'Transplant', 'Growth', 'Harvest']
synthetic_data = synthetic_data.drop(columns=cols_to_sync, errors='ignore')

sampled_rows = []
for crop in synthetic_data['Crop Name'].unique():
    idx = synthetic_data[synthetic_data['Crop Name'] == crop].index
    count = len(idx)
    orig_subset = data[data['Crop Name'] == crop][cols_to_sync]
    
    if not orig_subset.empty:
        sampled = orig_subset.sample(n=count, replace=True)
        sampled.index = idx
    else:
        sampled = pd.DataFrame(np.nan, index=idx, columns=cols_to_sync)
    sampled_rows.append(sampled)

sampled_df = pd.concat(sampled_rows)
synthetic_data = pd.concat([synthetic_data, sampled_df], axis=1)

# --- B. ĐẶC THÙ ĐỊA LÝ & KHÍ HẬU ---
geo_climate = data.groupby(['District', 'Season'])['Rainfall'].agg(Rain_min='min', Rain_max='max').reset_index()
dist_climate = data.groupby('District')['Rainfall'].agg(Dist_min='min', Dist_max='max').reset_index()

synthetic_data = synthetic_data.merge(geo_climate, on=['District', 'Season'], how='left')
synthetic_data = synthetic_data.merge(dist_climate, on='District', how='left')

global_min = data['Rainfall'].min()
global_max = data['Rainfall'].max()
synthetic_data['Rain_min'] = synthetic_data['Rain_min'].fillna(synthetic_data['Dist_min']).fillna(global_min)
synthetic_data['Rain_max'] = synthetic_data['Rain_max'].fillna(synthetic_data['Dist_max']).fillna(global_max)

synthetic_data['Rainfall'] = synthetic_data['Rainfall'].clip(
    lower=synthetic_data['Rain_min'], 
    upper=synthetic_data['Rain_max']
).round(2)
synthetic_data = synthetic_data.drop(columns=['Rain_min', 'Rain_max', 'Dist_min', 'Dist_max'])

# --- HELPER: Hàm an toàn chia cho 0 ---
def safe_div_robust(numerator, denominator, epsilon=1e-6):
    denom_safe = denominator.copy()
    mask_too_small = np.abs(denom_safe) < epsilon
    denom_safe[mask_too_small] = np.sign(denom_safe[mask_too_small]) * epsilon
    denom_safe[denom_safe == 0] = epsilon
    return numerator / denom_safe

# --- C. CHẶN BIÊN ÂM ---
non_negative_cols = [
    'Rainfall', 'Soil_Moisture_mm', 'Avg_Salinity_Index', 
    'Organic_Carbon', 'Nitrogen', 'sm_surface', 'sm_rootzone',
    'Wind_Mean', 'Wind_Max', 'Heat_Stress_Days',
    'EVI', 'LAI', 'FPAR' 
]
for col in non_negative_cols:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(lower=0)

def enforce_min_mean_max(df, col_min, col_mean, col_max):
    if {col_min, col_mean, col_max}.issubset(df.columns):
        mask_wrong = df[col_min] > df[col_max]
        cols = [col_min, col_max]
        df.loc[mask_wrong, cols] = df.loc[mask_wrong, cols].values[:, ::-1]
        df[col_mean] = df[col_mean].clip(lower=df[col_min], upper=df[col_max])
    return df

synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = enforce_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')

for col in ['Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

# --- D. CHUẨN HÓA CẢM BIẾN & TỶ LỆ ---
weather_cols = ['Min Temp', 'Avg Temp', 'Max Temp']
if all(col in synthetic_data.columns for col in weather_cols):
    synthetic_data[weather_cols] = synthetic_data[weather_cols].round(1)

ndvi_cols = [c for c in synthetic_data.columns if 'NDVI' in c]
for col in ndvi_cols:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)

if 'Rain_Temp_Ratio' in synthetic_data.columns:
    temp_safe = synthetic_data['Avg Temp'].copy()
    ratio = safe_div_robust(synthetic_data['Rainfall'], temp_safe)
    ratio[temp_safe <= 0] = 0 
    synthetic_data['Rain_Temp_Ratio'] = ratio 

if 'CN_Ratio' in synthetic_data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen'])

if 'Moisture_Ratio' in synthetic_data.columns:
    synthetic_data['sm_surface'] = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / synthetic_data['sm_surface'])

# --- E. LOGIC CỜ & RỦI RO ---
THRESHOLD_HEAT_DAYS = 43.0 
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns and 'Heat_Stress_Days' in synthetic_data.columns:
    mask_extreme = synthetic_data['is_extreme_Heat_Stress_Days'] == 1
    mask_update = mask_extreme & (synthetic_data['Heat_Stress_Days'] < THRESHOLD_HEAT_DAYS)
    synthetic_data.loc[mask_update, 'Heat_Stress_Days'] = np.random.uniform(THRESHOLD_HEAT_DAYS, 65.0, size=mask_update.sum())
    
    mask_normal = synthetic_data['is_extreme_Heat_Stress_Days'] == 0
    mask_update_normal = mask_normal & (synthetic_data['Heat_Stress_Days'] >= THRESHOLD_HEAT_DAYS)
    synthetic_data.loc[mask_update_normal, 'Heat_Stress_Days'] = np.random.uniform(0.0, 42.4, size=mask_update_normal.sum())

    synthetic_data['Heat_Stress_Days'] = (synthetic_data['Heat_Stress_Days'] * 2).round(0) / 2

THRESHOLD_WIND = 11.0
if 'is_extreme_Wind_Max' in synthetic_data.columns and 'Wind_Max' in synthetic_data.columns:
    mask_extreme_wind = synthetic_data['is_extreme_Wind_Max'] == 1
    mask_update = mask_extreme_wind & (synthetic_data['Wind_Max'] < THRESHOLD_WIND)
    synthetic_data.loc[mask_update, 'Wind_Max'] = np.random.uniform(THRESHOLD_WIND, 35.0, size=mask_update.sum())
    
    mask_normal_wind = synthetic_data['is_extreme_Wind_Max'] == 0
    mask_update_normal = mask_normal_wind & (synthetic_data['Wind_Max'] >= THRESHOLD_WIND)
    synthetic_data.loc[mask_update_normal, 'Wind_Max'] = np.random.uniform(2.0, 10.0, size=mask_update_normal.sum())
    
    if 'Wind_Mean' in synthetic_data.columns:
         mask_wrong_wind = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
         synthetic_data.loc[mask_wrong_wind, 'Wind_Mean'] = synthetic_data.loc[mask_wrong_wind, 'Wind_Max'] * 0.8
    
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)
    if 'Wind_Mean' in synthetic_data.columns:
        synthetic_data['Wind_Mean'] = synthetic_data['Wind_Mean'].round(2)

if 'Extreme_Heat_Risk' in synthetic_data.columns and 'Is_Extreme_Heat' in synthetic_data.columns:
    mask_contradiction = (synthetic_data['Is_Extreme_Heat'] == 1) & (synthetic_data['Extreme_Heat_Risk'] == 'Low Risk')
    if mask_contradiction.any():
        synthetic_data.loc[mask_contradiction, 'Extreme_Heat_Risk'] = 'High Risk' 
    mask_low_risk = synthetic_data['Extreme_Heat_Risk'] == 'Low Risk'
    synthetic_data.loc[mask_low_risk, 'Is_Extreme_Heat'] = 0

# --- F. TÍNH TOÁN YIELD SAU CÙNG ---
print("Đang tính toán lại Yield từ Production (đã sửa phân phối) và Area...")

# Thay 0 bằng 1 tạm thời để tránh chia cho 0
area_safe = synthetic_data['Area'].replace(0, 1)
synthetic_data['Yield'] = synthetic_data['Production'].astype(float) / area_safe.astype(float)

# Xử lý Soil Balance
soil_cols = ['Sand', 'Silt', 'Clay']
if all(col in synthetic_data.columns for col in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    total_soil = synthetic_data[soil_cols].sum(axis=1)
    mask_zero_sum = total_soil == 0
    if mask_zero_sum.any():
        synthetic_data.loc[mask_zero_sum, ['Sand', 'Silt', 'Clay']] = [33.33, 33.33, 33.34]
        total_soil[mask_zero_sum] = 100.0
        
    for col in soil_cols:
        synthetic_data[col] = (synthetic_data[col] / total_soil * 100)
        
    synthetic_data['Sand'] = synthetic_data['Sand'].round(2)
    synthetic_data['Silt'] = synthetic_data['Silt'].round(2)
    synthetic_data['Clay'] = (100.00 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2)
    
    mask_neg_clay = synthetic_data['Clay'] < 0
    mask_sand_dest = mask_neg_clay & (synthetic_data['Sand'] >= synthetic_data['Silt'])
    if mask_sand_dest.any():
        synthetic_data.loc[mask_sand_dest, 'Sand'] += synthetic_data.loc[mask_sand_dest, 'Clay']
    mask_silt_dest = mask_neg_clay & (synthetic_data['Sand'] < synthetic_data['Silt'])
    if mask_silt_dest.any():
        synthetic_data.loc[mask_silt_dest, 'Silt'] += synthetic_data.loc[mask_silt_dest, 'Clay']
    if mask_neg_clay.any():
        synthetic_data.loc[mask_neg_clay, 'Clay'] = 0

if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)

# --- G. ĐỒNG BỘ NHÃN PHÂN LOẠI ---
print("Đang đồng bộ lại Nhãn phân loại...")
if 'Dominant_Soil_Texture' in synthetic_data.columns:
    conditions_soil = [
        (synthetic_data['Clay'] >= 40),
        (synthetic_data['Sand'] >= 50),
        (synthetic_data['Silt'] >= 50)
    ]
    choices_soil = ['Clayey', 'Sandy', 'Silty']
    synthetic_data['Dominant_Soil_Texture'] = np.select(conditions_soil, choices_soil, default='Loamy')

if 'pH_Suitability' in synthetic_data.columns:
    conditions_ph = [
        (synthetic_data['pH'] < 5.5),
        (synthetic_data['pH'] > 7.5)
    ]
    choices_ph = ['Acidic', 'Alkaline']
    synthetic_data['pH_Suitability'] = np.select(conditions_ph, choices_ph, default='Optimal')

# ---------------------------------------------------------
# 6. LƯU FILE
# ---------------------------------------------------------
output_file = 'Agri_Data_CTGAN.csv'
synthetic_data.to_csv(output_file, index=False)
print(f"Hoàn tất! Dữ liệu CTGAN (Siêu tinh chỉnh) đã được lưu tại: {output_file}")
print("Phiên bản: Tuned + Quantile Matching (Area, Production) + Fix Dtypes")

Đang đọc dữ liệu...
Đã loại bỏ cột 'Yield' khỏi dữ liệu huấn luyện.
Đang khởi tạo CTGAN với cấu hình tối ưu...
Bắt đầu huấn luyện... (Quá trình này có thể mất vài phút)


Gen. (-00.29) | Discrim. (-00.42): 100%|██████████| 500/500 [06:44<00:00,  1.24it/s]


Đang sinh dữ liệu theo chiến lược 'Reject Sampling'...
Đã chọn lọc 20000 mẫu chất lượng cao.
Đang áp dụng thuật toán Quantile Matching để sửa phân phối Area & Production...
Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...
Đang tính toán lại Yield từ Production (đã sửa phân phối) và Area...
Đang đồng bộ lại Nhãn phân loại...
Hoàn tất! Dữ liệu CTGAN (Siêu tinh chỉnh) đã được lưu tại: Agri_Data_CTGAN.csv
Phiên bản: Tuned + Quantile Matching (Area, Production) + Fix Dtypes


: 